In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableBranch, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv()

loader = PyPDFLoader("../sample.pdf")
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.split_documents(pages)

In [2]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(k=10)

/tmp/ipykernel_51105/3297151316.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=os.environ["GEMINI_API_KEY"])
output_parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

E0000 00:00:1759060488.218490   51105 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [4]:
router_prompt = PromptTemplate.from_template(
    """사용자의 질문이 주어진 '문서'의 내용과 관련이 있는지, 아니면 '일반 대화'인지 판단하세요.
'문서 관련' 또는 '일반 대화' 둘 중 하나로만 답변하세요.

사용자 질문: {question}"""
)
router = router_prompt | llm | output_parser

In [5]:
rag_prompt = PromptTemplate.from_template(
    """아래 제공된 문서를 참고해서 질문에 답변해라.
문서가 직접적으로 답을 주지 않더라도, 관련된 정보를 최대한 설명해라.

문서:
{context}

질문: {question}
"""
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | output_parser
)

In [6]:
general_prompt = PromptTemplate.from_template("다음 질문에 친절하게 답변해줘: {question}")
general_chain = general_prompt | llm | output_parser

In [7]:
branch = RunnableBranch(
    (lambda x: "문서 관련" in x['topic'], rag_chain),
    general_chain
)

In [8]:
full_chain = {"topic": router, "question": lambda x: x['question']} | branch

In [13]:
print(full_chain.invoke({"question": "이 문서를 정리해줘."}))
print()
print(full_chain.invoke({"question": "오늘 하루 어땠어?"}))

제공된 문서는 어텐션(Attention) 메커니즘, 특히 'Scaled Dot-Product Attention'과 'Multi-Head Attention'을 중심으로 설명하며, 이들이 어떻게 시퀀스 내의 정보를 처리하고 장거리 의존성을 학습하는 데 사용되는지를 다룹니다.

다음은 문서의 주요 내용을 정리한 것입니다:

1.  **어텐션(Attention)의 정의 및 기본 개념 (3.2 Attention)**
    *   어텐션 함수는 **쿼리(query)**와 **키-값(key-value) 쌍의 집합**을 입력으로 받아 하나의 **출력**으로 매핑합니다.
    *   출력은 **값(values)의 가중치 합**으로 계산됩니다. 이때 각 값에 할당되는 가중치는 쿼리와 해당 키의 호환성 함수에 의해 결정됩니다.
    *   트랜스포머(Transformer) 모델에서 장거리 의존성(long-distance dependencies) 학습 시 연산 수를 줄이지만, 평균화로 인한 '유효 해상도 감소'는 **Multi-Head Attention**으로 보완됩니다.
    *   **Self-attention (또는 Intra-attention)**은 단일 시퀀스 내의 서로 다른 위치 간의 관계를 파악하여 시퀀스 표현을 계산하는 어텐션 메커니즘입니다. 읽기 이해, 요약, 텍스트 함의 등 다양한 자연어 처리(NLP) 작업에서 성공적으로 사용되었습니다.

2.  **Scaled Dot-Product Attention (3.2.1 Scaled Dot-Product Attention)**
    *   문서에서 소개하는 특정 어텐션 메커니즘입니다 (Figure 2, 왼쪽).
    *   **입력:** `dk` 차원의 쿼리(queries, Q)와 키(keys, K), `dv` 차원의 값(values, V)으로 구성됩니다.
    *   **계산 과정:**
        1.  쿼리와 모든 키의 내적(dot product)을 계산합니다.
        2.  각 결과를 `√dk